In [ ]:
import os
import pandas as pd
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torchvision import models
import torch.nn as nn
import torch.optim as optim
from torchaudio.transforms import MFCC
from collections import defaultdict
from tqdm import tqdm
from scipy.linalg import sqrtm
from sklearn.metrics import pairwise_distances
from collections import defaultdict
from scipy.spatial.distance import jensenshannon
from sklearn.cluster import KMeans

In [ ]:
df_birds = pd.read_csv('df_birds.csv')
df_birds = df_birds.dropna(subset=['class_id'])
df_birds.head()

In [ ]:
import torch
from torch.utils.data import Dataset
import librosa

class BirdDataset(Dataset):
    def __init__(self, df, mfcc_transform, sample_rate=22050):
        self.df = df.reset_index(drop=True)
        self.mfcc_transform = mfcc_transform
        self.target_sr = sample_rate

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]
        path = row['relative_path']
        label = int(row['class'])
        waveform, sr = librosa.load(path, sr=self.target_sr,mono=True)
        waveform = torch.from_numpy(waveform).unsqueeze(0)
        mfcc = self.mfcc_transform(waveform)

        return mfcc, label



In [ ]:
train_df, val_df = train_test_split(df_birds, test_size=0.2, random_state=42,)

In [ ]:
mfcc_transform = MFCC(
    sample_rate=SAMPLE_RATE,
    n_mfcc=N_MFCC,
    melkwargs={
        'n_fft': 1024,
        'hop_length': 256,
        'n_mels': N_MFCC
    }
)

train_dataset = BirdDataset(train_df, mfcc_transform)
val_dataset = BirdDataset(val_df, mfcc_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, )
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True, )

In [ ]:
print(train_dataset[0][0].shape,train_dataset[0][1])

In [ ]:
#ResNet50
num_classes = 12
model = models.resnet50(pretrained=True)
model.conv1 = nn.Conv2d(
    in_channels=1,
    out_channels=model.conv1.out_channels,
    kernel_size=model.conv1.kernel_size,
    stride=model.conv1.stride,
    padding=model.conv1.padding,
    bias=False
)

model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

best_val_acc = 0.0
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(DEVICE)
            labels = labels.to(DEVICE)
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
    val_acc = correct / len(val_loader.dataset)

    print(f"Epoch {epoch}/{NUM_EPOCHS} - Train Loss: {epoch_loss:.4f} - Val Acc: {val_acc:.4f}")

    # save
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/content/drive/MyDrive/best_resnet50.pth")
        print(f"  Saved new best model (Val Acc: {best_val_acc:.4f})")

In [ ]:
weights_path = "best_resnet50.pth"
gen_dir = "output/"
df_csv = "df_birds.csv"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sample_rate = 22050
n_mfcc = 80
mfcc_kwargs = {'n_fft': 1024, 'hop_length': 256, 'n_mels': 80}

# Load data and create mapping
df = pd.read_csv(df_csv)
df.dropna(inplace=True)

mapping = {}
for rel_path, cls in zip(df['relative_path'], df['class']):
   base = os.path.splitext(os.path.basename(rel_path))[0]
   mapping[base] = int(cls)
   mapping[f"{base}_resampled"] = int(cls)

# Load and configure model
model = models.resnet50(pretrained=True)
model.conv1 = nn.Conv2d(1, model.conv1.out_channels,
                       kernel_size=model.conv1.kernel_size,
                       stride=model.conv1.stride,
                       padding=model.conv1.padding,
                       bias=False)
model.fc = nn.Linear(model.fc.in_features, 12)
model.load_state_dict(torch.load(weights_path, map_location=device))
model.to(device).eval()

# Initialize MFCC transform
mfcc_transform = MFCC(sample_rate=sample_rate,
                     n_mfcc=n_mfcc,
                     melkwargs=mfcc_kwargs)

# Initialize tracking variables
class_totals = defaultdict(int)
class_correct = defaultdict(int)
overall_total = 0
overall_correct = 0

# Process files and evaluate
for fname in sorted(os.listdir(gen_dir)):
   if not fname.endswith('.wav'):
       continue
   
   base = os.path.splitext(fname)[0]
   if base not in mapping:
       continue
   
   true_label = mapping[base]
   
   # Load and preprocess audio
   wav, sr = torchaudio.load(os.path.join(gen_dir, fname))
   if sr != sample_rate:
       wav = torchaudio.functional.resample(wav, sr, sample_rate)
   
   wav = wav.mean(dim=0, keepdim=True)
   mfcc = mfcc_transform(wav)
   mfcc = mfcc.unsqueeze(1).to(device)
   
   # Predict
   with torch.no_grad():
       logits = model(mfcc)
   pred = logits.argmax(dim=1).item()
   
   # Update metrics
   overall_total += 1
   if pred == true_label:
       overall_correct += 1
       class_correct[true_label] += 1
   class_totals[true_label] += 1

# Print results
print(f"accuracy: {overall_correct}/{overall_total} = {overall_correct/overall_total*100:.2f}%\n")
print("class accuracy:")

for cls in range(12):
   tot = class_totals.get(cls, 0)
   corr = class_correct.get(cls, 0)
   if tot > 0:
       acc = corr / tot * 100
       print(f"  class {cls:2d}: {corr}/{tot} = {acc:.2f}%")
   else:
       print(f"  class {cls:2d}: no samples")


## FAD

In [ ]:
class vggfeature(nn.module):

   def __init__(self):
       super().__init__()
       import torchvision.models as models
       vgg = models.vgg16(pretrained=true)

       self.features = nn.sequential(
           nn.conv2d(1, 64, kernel_size=3, padding=1),
           nn.relu(inplace=true),
           *list(vgg.features)[1:]
       )

       self.avgpool = nn.adaptiveavgpool2d((7, 7))
       self.classifier = nn.sequential(
           nn.linear(512 * 7 * 7, 4096),
           nn.relu(true),
           nn.dropout(),
           nn.linear(4096, 1024),
       )

       self.eval()

   def forward(self, x):
       x = self.features(x)
       x = self.avgpool(x)
       x = torch.flatten(x, 1)
       x = self.classifier(x)
       return x

class audioproc:
   def __init__(self, sample_rate=22050, n_mels=128, n_fft=1024, hop_length=256):
       self.sample_rate = sample_rate
       self.mel_transform = torchaudio.transforms.melspectrogram(
           sample_rate=sample_rate,
           n_mels=n_mels,
           n_fft=n_fft,
           hop_length=hop_length
       )

   def preprocess_audio(self, audio_path, target_length=3.0):
       try:
           waveform, sr = torchaudio.load(audio_path)

           if sr != self.sample_rate:
               resampler = torchaudio.transforms.resample(sr, self.sample_rate)
               waveform = resampler(waveform)

           if waveform.shape[0] > 1:
               waveform = torch.mean(waveform, dim=0, keepdim=true)

           target_samples = int(target_length * self.sample_rate)
           if waveform.shape[1] > target_samples:
               waveform = waveform[:, :target_samples]
           elif waveform.shape[1] < target_samples:
               padding = target_samples - waveform.shape[1]
               waveform = torch.nn.functional.pad(waveform, (0, padding))

           mel_spec = self.mel_transform(waveform)

           mel_spec_db = torchaudio.transforms.amplitudetodb()(mel_spec)
           mel_spec_norm = (mel_spec_db - mel_spec_db.mean()) / (mel_spec_db.std() + 1e-8)

           return mel_spec_norm.unsqueeze(0)

       except exception as e:
           print(f"error processing {audio_path}: {e}")
           return none

class multifad:

   def __init__(self, device='cuda' if torch.cuda.is_available() else 'cpu'):
       self.device = device
       self.feat_extractor = vggfeature().to(device)
       self.audio_proc = audioproc()

   def calc_fad(self, real_feat, gen_feat):
       if len(real_feat) == 0 or len(gen_feat) == 0:
           return float('inf')

       mu_real = np.mean(real_feat, axis=0)
       mu_gen = np.mean(gen_feat, axis=0)

       sigma_real = np.cov(real_feat, rowvar=false)
       sigma_gen = np.cov(gen_feat, rowvar=false)

       diff = mu_real - mu_gen

       try:
           sqrt_sigma = sqrtm(sigma_real @ sigma_gen)
           if np.iscomplexobj(sqrt_sigma):
               sqrt_sigma = sqrt_sigma.real
       except:
           eps = 1e-6
           sqrt_sigma = sqrtm((sigma_real + eps * np.eye(sigma_real.shape[0])) @
                             (sigma_gen + eps * np.eye(sigma_gen.shape[0])))
           if np.iscomplexobj(sqrt_sigma):
               sqrt_sigma = sqrt_sigma.real

       fad = np.dot(diff, diff) + np.trace(sigma_real + sigma_gen - 2 * sqrt_sigma)

       return fad

   def multi_fad(self, gen_dir, df_csv, real_base, max_files=none):

       df = pd.read_csv(df_csv)
       df.dropna(inplace=true)

       file_class = {}
       class_real = defaultdict(list)

       for _, row in df.iterrows():
           rel_path = row['relative_path']
           class_id = int(row['class'])

           base = os.path.splitext(os.path.basename(rel_path))[0]
           file_class[base] = class_id
           file_class[f"{base}_resampled"] = class_id

           full_path = os.path.join(real_base, rel_path)
           if os.path.exists(full_path):
               class_real[class_id].append(full_path)

       print(f"loaded mapping for {len(file_class)} files")
       print(f"real audio paths for {len(class_real)} classes")

       class_gen = defaultdict(list)

       for fname in os.listdir(gen_dir):
           if not fname.endswith('.wav'):
               continue

           base = os.path.splitext(fname)[0]
           if base in file_class:
               class_id = file_class[base]
               gen_path = os.path.join(gen_dir, fname)
               class_gen[class_id].append(gen_path)

       print(f"generated audio files grouped into {len(class_gen)} classes")

       print("computing multi-class fad")

       class_fads = {}
       class_counts = {}
       valid_classes = []

       for class_id in sorted(set(class_real.keys()) | set(class_gen.keys())):
           print(f"processing class {class_id}")

           if class_id not in class_gen:
               print(f"no generated audio for class {class_id}")
               class_fads[class_id] = float('inf')
               class_counts[class_id] = 0
               continue

           if class_id not in class_real:
               print(f"no real audio for class {class_id}")
               class_fads[class_id] = float('inf')
               class_counts[class_id] = 0
               continue

           gen_paths = class_gen[class_id]
           real_paths = class_real[class_id]

           print(f"  generated files: {len(gen_paths)}")
           print(f"  real files: {len(real_paths)}")

           try:
               gen_feat = self.extract_feat(gen_paths, max_files)

               real_feat = self.extract_feat(real_paths, max_files)

               fad_score = self.calc_fad(real_feat, gen_feat)

               class_fads[class_id] = fad_score
               class_counts[class_id] = len(gen_feat)
               valid_classes.append(class_id)

               print(f"class {class_id} fad: {fad_score:.4f} (generated: {len(gen_feat)}, real: {len(real_feat)})")

           except exception as e:
               print(f"error processing class {class_id}: {e}")
               import traceback
               traceback.print_exc()
               class_fads[class_id] = float('inf')
               class_counts[class_id] = 0

       valid_fads = [class_fads[class_id] for class_id in valid_classes if class_fads[class_id] != float('inf')]
       avg_fad = np.mean(valid_fads) if valid_fads else float('inf')

       print("multi-class fad results")

       print("per-class fad:")
       for class_id in sorted(class_fads.keys()):
           fad = class_fads[class_id]
           count = class_counts[class_id]
           if fad != float('inf'):
               print(f"  class {class_id:2d}: {fad:8.4f} ({count:3d} samples)")
           else:
               print(f"  class {class_id:2d}: failed    ({count:3d} samples)")

       print(f"valid classes: {len(valid_classes)}/12")
       print(f"average fad: {avg_fad:.4f}")
       print(f"total samples: {sum(class_counts.values())}")

       return class_fads, avg_fad, class_counts

   def extract_feat(self, file_paths, max_files=none):
       features = []

       if max_files:
           file_paths = file_paths[:max_files]

       with torch.no_grad():
           for audio_path in file_paths:
               mel_spec = self.audio_proc.preprocess_audio(audio_path)

               if mel_spec is not none:
                   mel_spec = mel_spec.to(self.device)

                   feature = self.feat_extractor(mel_spec)
                   features.append(feature.cpu().numpy())

       if len(features) == 0:
           raise valueerror(f"no valid audio files found in provided paths")

       return np.vstack(features)


def multi_fad_df(gen_dir=gen_dir,
                                          df_csv=df_csv,
                                          real_base=real_base,
                                          max_files=none):

   fad_calc = multifad()

   class_fads, avg_fad, class_counts = fad_calc.multi_fad(
       gen_dir, df_csv, real_base, max_files
   )

   return class_fads, avg_fad, class_counts

if __name__ == "__main__":
   try:

       class_fads, avg_fad, class_counts = multi_fad_df(
           gen_dir="/content/output/",
           df_csv="df_birds.csv",
           real_base="/content/data/",
           max_files=50
       )

       print(f"final results:")
       print(f"average fad across all classes: {avg_fad:.4f}")
       valid_count = len([k for k, v in class_fads.items() if v != float('inf')])
       print(f"valid classes processed: {valid_count}/12")

       valid_fads = [v for v in class_fads.values() if v != float('inf')]
       if valid_fads:
           print(f"best class fad: {min(valid_fads):.4f}")
           print(f"worst class fad: {max(valid_fads):.4f}")
           print(f"fad std dev: {np.std(valid_fads):.4f}")

   except exception as e:
       print(f"error computing multi-class fad: {e}")
       import traceback
       traceback.print_exc()

## JSD and NDB

In [ ]:
class multindb:

   def __init__(self, sample_rate=22050, min_duration=0.1, n_bins=20):
       self.sample_rate = sample_rate
       self.min_samples = int(min_duration * sample_rate)
       self.feat_len = 10
       self.n_bins = n_bins

   def extract_feat(self, audio_path, pad_short=true):
       try:
           waveform, sr = torchaudio.load(audio_path)

           if sr != self.sample_rate:
               resampler = torchaudio.transforms.resample(sr, self.sample_rate)
               waveform = resampler(waveform)

           if waveform.shape[0] > 1:
               waveform = torch.mean(waveform, dim=0)

           audio = waveform.numpy()

           if len(audio) < self.min_samples:
               if pad_short:
                   padding = self.min_samples - len(audio)
                   audio = np.pad(audio, (0, padding), mode='constant', constant_values=0)
               else:
                   return none

           features = []

           features.extend([
               np.mean(audio),
               np.std(audio),
               np.max(np.abs(audio)),
               np.mean(np.abs(audio)),
               len(np.where(np.diff(np.sign(audio)))[0]) / len(audio)
           ])

           try:
               n_fft = 512
               if len(audio) < n_fft:
                   audio_padded = np.pad(audio, (0, n_fft - len(audio)), mode='constant')
               else:
                   audio_padded = audio

               fft = np.fft.fft(audio_padded, n=n_fft)
               magnitude = np.abs(fft[:n_fft//2])

               features.extend([
                   np.mean(magnitude),
                   np.std(magnitude),
                   np.argmax(magnitude) / len(magnitude),
                   np.sum(magnitude),
               ])

               freqs = np.arange(len(magnitude))
               if np.sum(magnitude) > 0:
                   spec_cent = np.sum(freqs * magnitude) / np.sum(magnitude)
                   spec_cent = spec_cent / len(magnitude)
               else:
                   spec_cent = 0.0

               features.append(spec_cent)

           except exception as e:
               features.extend([0.0, 0.0, 0.0, 0.0, 0.0])

           features = features[:self.feat_len]
           while len(features) < self.feat_len:
               features.append(0.0)

           features = np.array(features, dtype=np.float32)
           features = np.nan_to_num(features, nan=0.0, posinf=1.0, neginf=0.0)

           return features

       except exception as e:
           return np.zeros(self.feat_len, dtype=np.float32)

   def extract_list(self, file_paths, max_files=none, pad_short=true):
       if max_files:
           file_paths = file_paths[:max_files]

       features = []

       for audio_path in file_paths:
           feature = self.extract_feat(audio_path, pad_short=pad_short)

           if feature is not none:
               if len(feature) == self.feat_len:
                   features.append(feature)

       if len(features) == 0:
           raise valueerror(f"no valid features extracted from file list")

       feat_array = np.array(features)
       return feat_array

   def calc_jsd(self, real_feat, gen_feat):
       jsds = []

       for dim in range(real_feat.shape[1]):
           real_data = real_feat[:, dim]
           gen_data = gen_feat[:, dim]

           min_val = min(real_data.min(), gen_data.min())
           max_val = max(real_data.max(), gen_data.max())

           if max_val == min_val:
               max_val = min_val + 1e-6

           bins = np.linspace(min_val, max_val, self.n_bins + 1)

           hist_real, _ = np.histogram(real_data, bins=bins, density=true)
           hist_gen, _ = np.histogram(gen_data, bins=bins, density=true)

           hist_real = hist_real / np.sum(hist_real) + 1e-10
           hist_gen = hist_gen / np.sum(hist_gen) + 1e-10

           hist_real = hist_real / np.sum(hist_real)
           hist_gen = hist_gen / np.sum(hist_gen)

           jsd = jensenshannon(hist_real, hist_gen) ** 2
           jsds.append(jsd)

       jsd_mean = np.mean(jsds)
       return jsd_mean

   def calc_ndb(self, real_feat, gen_feat):
       kmeans = kmeans(n_clusters=self.n_bins, random_state=42, n_init=10)
       try:
           cluster_labels = kmeans.fit_predict(real_feat)
       except exception as e:
           return self._calc_ndb_alt(real_feat, gen_feat)

       real_bin_counts = np.bincount(cluster_labels, minlength=self.n_bins)

       gen_cluster_labels = kmeans.predict(gen_feat)
       gen_bin_counts = np.bincount(gen_cluster_labels, minlength=self.n_bins)

       total_real = len(real_feat)
       total_gen = len(gen_feat)

       real_prop = real_bin_counts / total_real
       gen_prop = gen_bin_counts / total_gen

       threshold = 0.05
       ndb_count = 0
       for i in range(self.n_bins):
           if real_bin_counts[i] < 3:
               continue

           if real_prop[i] > 0:
               ratio = gen_prop[i] / real_prop[i]
               if ratio > (1 + threshold) or ratio < threshold:
                   ndb_count += 1
           elif gen_prop[i] > threshold:
               ndb_count += 1

       return ndb_count

   def _calc_ndb_alt(self, real_feat, gen_feat):
       ndb_count = 0

       for dim in range(real_feat.shape[1]):
           real_data = real_feat[:, dim]
           gen_data = gen_feat[:, dim]

           min_val = min(real_data.min(), gen_data.min())
           max_val = max(real_data.max(), gen_data.max())

           if max_val == min_val:
               continue

           bins = np.linspace(min_val, max_val, self.n_bins + 1)

           real_hist, _ = np.histogram(real_data, bins=bins)
           gen_hist, _ = np.histogram(gen_data, bins=bins)

           real_prop = real_hist / len(real_data)
           gen_prop = gen_hist / len(gen_data)

           threshold = 0.05
           for i in range(len(real_hist)):
               if real_prop[i] > 0:
                   ratio = gen_prop[i] / real_prop[i] if real_prop[i] > 0 else float('inf')
                   if ratio > (1 + threshold) or ratio < threshold:
                       ndb_count += 1
               elif gen_prop[i] > threshold:
                   ndb_count += 1

       ndb_final = ndb_count // real_feat.shape[1]
       return ndb_final

   def multi_ndb_jsd(self, gen_dir, df_csv, real_base, max_files=none):
       df = pd.read_csv(df_csv)
       df.dropna(inplace=true)

       file_class = {}
       class_real = defaultdict(list)

       for _, row in df.iterrows():
           rel_path = row['relative_path']
           class_id = int(row['class'])

           base = os.path.splitext(os.path.basename(rel_path))[0]
           file_class[base] = class_id
           file_class[f"{base}_resampled"] = class_id

           full_path = os.path.join(real_base, rel_path)
           if os.path.exists(full_path):
               class_real[class_id].append(full_path)

       class_gen = defaultdict(list)

       for fname in os.listdir(gen_dir):
           if not fname.endswith('.wav'):
               continue

           base = os.path.splitext(fname)[0]
           if base in file_class:
               class_id = file_class[base]
               gen_path = os.path.join(gen_dir, fname)
               class_gen[class_id].append(gen_path)

       class_ndbs = {}
       class_jsds = {}
       class_counts = {}
       valid_classes = []

       for class_id in sorted(set(class_real.keys()) | set(class_gen.keys())):
           if class_id not in class_gen or class_id not in class_real:
               class_ndbs[class_id] = float('inf')
               class_jsds[class_id] = float('inf')
               class_counts[class_id] = 0
               continue

           gen_paths = class_gen[class_id]
           real_paths = class_real[class_id]

           try:
               gen_feat = self.extract_list(gen_paths, max_files)
               real_feat = self.extract_list(real_paths, max_files)

               ndb_score = self.calc_ndb(real_feat, gen_feat)
               jsd_score = self.calc_jsd(real_feat, gen_feat)

               class_ndbs[class_id] = ndb_score
               class_jsds[class_id] = jsd_score
               class_counts[class_id] = len(gen_feat)
               valid_classes.append(class_id)

           except exception as e:
               class_ndbs[class_id] = float('inf')
               class_jsds[class_id] = float('inf')
               class_counts[class_id] = 0

       valid_ndbs = [class_ndbs[class_id] for class_id in valid_classes if class_ndbs[class_id] != float('inf')]
       valid_jsds = [class_jsds[class_id] for class_id in valid_classes if class_jsds[class_id] != float('inf')]

       avg_ndb = np.mean(valid_ndbs) if valid_ndbs else float('inf')
       avg_jsd = np.mean(valid_jsds) if valid_jsds else float('inf')
       total_ndb = sum(valid_ndbs) if valid_ndbs else float('inf')

       print("multi-class ndb & jsd results")
       print("per-class results:")
       for class_id in sorted(class_ndbs.keys()):
           ndb = class_ndbs[class_id]
           jsd = class_jsds[class_id]
           count = class_counts[class_id]
           if ndb != float('inf'):
               print(f"  class {class_id:2d}: ndb={ndb:2.0f}, jsd={jsd:8.6f} ({count:3d} samples)")
           else:
               print(f"  class {class_id:2d}: failed ({count:3d} samples)")

       print(f"valid classes: {len(valid_classes)}")
       print(f"total ndb (all classes): {total_ndb:.0f}")
       print(f"average ndb per class: {avg_ndb:.2f}")
       print(f"average jsd: {avg_jsd:.6f}")

       return class_ndbs, class_jsds, total_ndb, avg_jsd, class_counts

def multi_ndb_jsd(gen_dir="/content/output/",
                df_csv="df_birds.csv",
                real_base="/content/data/",
                max_files=50,
                n_bins=20):
   calc = multindb(n_bins=n_bins)

   class_ndbs, class_jsds, total_ndb, avg_jsd, class_counts = calc.multi_ndb_jsd(
       gen_dir, df_csv, real_base, max_files
   )

   return class_ndbs, class_jsds, total_ndb, avg_jsd, class_counts

if __name__ == "__main__":
   try:
       class_ndbs, class_jsds, total_ndb, avg_jsd, class_counts = multi_ndb_jsd(
           max_files=100,
           n_bins=10
       )

       print(f"final summary:")
       print(f"total ndb across all classes: {total_ndb:.0f}")
       print(f"average jsd across all classes: {avg_jsd:.6f}")

   except exception as e:
       print(f"error: {e}")